# Python Idioms & Syntax

This is a living reference — not a complete Python tutorial, but a collection
of idiomatic patterns that appear across the `protoform` codebase.

Each entry covers *what* a pattern does, *why* it exists, and *where* you'll
see it used in the real world.  The goal is that after reading an entry you
can recognise the pattern anywhere and use it yourself with confidence.

---

| Pattern | Where it appears |
|---|---|
| `sorted(set(...))` | `CharTokenizer.__init__` |
| Dict comprehension | `CharTokenizer.__init__` |
| `enumerate` | `CharTokenizer.__init__` |
| List as indexed lookup | `CharTokenizer.decode` |
| Type hints | Throughout `protoform` |
| `Protocol` | `protoform.protocols` |
| PEP 695 generics `[CorpusT]` | `protoform.protocols.Tokenizer` |

## `sorted(set(iterable))`

**What it does:** produces a sorted list of unique elements from any iterable.

**Why two steps?**

- `set(iterable)` removes duplicates in O(n) — sets store only one copy of each value.
- `sorted(...)` turns the set into a list in a guaranteed, deterministic order.

Both steps are needed.  A `set` alone is unordered — iterating it on two
different runs (or Python versions) may give different orders.  `sorted`
makes the result stable and reproducible, which is critical when you're
building a vocabulary where token ID 0 must always mean the same character.

In [ ]:
text = "banana"

# set removes duplicates, but order is not guaranteed
print(set(text))  # e.g. {'a', 'b', 'n'} — order may vary

# sorted gives a stable, alphabetical list
print(sorted(set(text)))  # ['a', 'b', 'n'] — always this order

# In CharTokenizer this becomes the vocabulary
corpus = "hello world"
vocab = sorted(set(corpus))
print(vocab)  # [' ', 'd', 'e', 'h', 'l', 'o', 'r', 'w']

## Dict comprehension

**What it does:** builds a dict from an iterable in a single expression.

```python
{key_expr: value_expr for item in iterable}
```

This is the dict equivalent of a list comprehension.  It reads as:
*"for each item, produce this key mapped to this value"*.

It is faster and more readable than building an empty dict and calling
`.update()` or assigning in a loop.

In [ ]:
# The explicit loop version — correct but verbose
vocab = ["a", "b", "n"]
stoi = {}
for i, c in enumerate(vocab):
    stoi[c] = i
print(stoi)

# The comprehension version — same result, one line
# The type hint documents exactly what this structure is
stoi: dict[str, int] = {c: i for i, c in enumerate(vocab)}
print(stoi)

# You can also filter inside a comprehension
only_ab: dict[str, int] = {c: i for i, c in enumerate(vocab) if c != "n"}
print(only_ab)

## `enumerate`

**What it does:** wraps any iterable and yields `(index, value)` pairs.

**Why not just use `range(len(...))`?**

The range-and-index pattern works but forces you to write `vocab[i]` to
get the value — you're indexing into the list instead of just iterating it.
`enumerate` gives you both the counter *and* the value directly, which is
cleaner and works on any iterable (not just lists).

In [ ]:
vocab = ["a", "b", "n"]

# The old way — index into the list manually
for i in range(len(vocab)):
    print(i, vocab[i])

print()

# The Pythonic way — enumerate gives (index, value) directly
for i, c in enumerate(vocab):
    print(i, c)

print()

# You can start counting from any number
for i, c in enumerate(vocab, start=1):
    print(i, c)

## Putting it together — `CharTokenizer.__init__`

These three patterns combine into the heart of the tokenizer.

Notice that `itos` (the reverse mapping) is **not a dict** — it's just the
`chars` list itself.  Since token IDs are contiguous integers starting at 0,
the list index *is* the token ID.  `chars[i]` gives the character for ID `i`
directly, with no extra memory allocation.

In [ ]:
corpus = "hello world"

# Step 1: unique, sorted characters — the vocabulary
# This list is the ground truth: position = token ID
chars = sorted(set(corpus))
print("vocab:", chars)

# Step 2: char -> id  (used by encode)
# A dict is needed here because lookup is by character, not by position
stoi = {c: i for i, c in enumerate(chars)}
print("stoi:", stoi)

# Step 3: id -> char  (used by decode)
# No dict needed — chars[i] already gives the character for token ID i
print("chars[3]:", chars[3])  # same as itos[3] would give

# Encoding: char -> id via the dict
print("encode 'hello':", [stoi[c] for c in "hello"])

# Decoding: id -> char via the list
print("decode [3,4,5,5,6]:", "".join(chars[i] for i in [3, 4, 5, 5, 6]))

## Type hints

**What they do:** annotate variables and function signatures with their expected types.

Type hints are not enforced at runtime — Python ignores them during execution.
Their value is entirely *static*: tools like pyright and mypy read them before
the code runs and flag mismatches.  Think of them as a contract you write for
your tools and your future self.

```python
# Without hints — works, but the reader must infer types
def encode(text):
    ...

# With hints — intent is explicit, tools can verify callers
def encode(text: str) -> list[int]:
    ...
```

**Built-in collection types** use lowercase since Python 3.9:
`list[str]`, `dict[str, int]`, `tuple[float, float]`, `set[str]`.

**Union types** use `|` since Python 3.10:
`str | None` replaces the older `Optional[str]`.

In [ ]:
# Variable annotation
vocab_size: int = 65
chars: list[str] = sorted(set("hello world"))
stoi: dict[str, int] = {c: i for i, c in enumerate(chars)}


# Function annotation — inputs and output are explicit
def encode(text: str, mapping: dict[str, int]) -> list[int]:
    return [mapping[c] for c in text]


# Union — the argument can be a str or None
def greet(name: str | None = None) -> str:
    return f"hello {name}" if name else "hello"


print(encode("hello", stoi))
print(greet())
print(greet("world"))

# Pyright verifies these statically — try passing an int to encode() in your IDE
# and the type checker will flag it before you ever run the code

### `str | None = None` — two separate `None`s

This pattern appears constantly in Python and confuses almost everyone
the first time they see it:

```python
def greet(name: str | None = None) -> str:
```

There are **two independent `None`s** doing different jobs:

| | What it is | What it controls |
|---|---|---|
| `str \| None` | the **type** | what values are *allowed* |
| `= None` | the **default** | what value is used when the caller passes *nothing* |

They are independent — you can mix them freely:

In [ ]:
# 1. No default — caller MUST pass str or None explicitly
def greet_required(name: str | None) -> str:
    return f"hello {name}" if name else "hello"


# 2. Default is a string — caller may omit, type must be str
def greet_default(name: str = "world") -> str:
    return f"hello {name}"


# 3. Default is None — caller may omit, absent state is None
def greet_optional(name: str | None = None) -> str:
    return f"hello {name}" if name else "hello"


# 4. Default is a string but None is also valid
def greet_flexible(name: str | None = "world") -> str:
    return f"hello {name}" if name else "hello"


print(greet_required("Alice"))  # must pass something
print(greet_required(None))  # None is valid
print(greet_default())  # omit → uses 'world'
print(greet_optional())  # omit → uses None → 'hello'
print(greet_flexible())  # omit → uses 'world'
print(greet_flexible(None))  # pass None → 'hello'

# The reason str | None = None is so common:
# if your DEFAULT is None, then None MUST be in the type —
# otherwise pyright flags the default as a type violation.
#
# def broken(name: str = None) -> str:  # pyright error:
#     ...                               # None not assignable to str

## `Protocol` — structural subtyping

**What it does:** defines an interface by *shape* rather than inheritance.

In classical OOP you'd write `class CharTokenizer(BaseTokenizer)` and inherit
from a base class.  Python's `Protocol` takes the opposite approach — called
**duck typing with type-checker support**:

> *If it has the right methods, it satisfies the interface — no explicit
> inheritance needed.*

This is called **structural subtyping** (as opposed to nominal subtyping).
It is the Python way: prefer composition and interfaces over inheritance hierarchies.

**`@runtime_checkable` vs static checking**

While `@runtime_checkable` allows `isinstance()` checks during execution,
the true power of `Protocol` happens *before the code even runs* — tools like
pyright or mypy catch interface mismatches statically.  If you pass an object
that is missing `decode`, the type checker flags it at write time, not at
runtime when a user hits an error.

In [ ]:
from typing import Protocol, runtime_checkable


@runtime_checkable
class Greetable(Protocol):
    def greet(self) -> str: ...


# No inheritance — just implement the method
class Dog:
    def greet(self) -> str:
        return "woof"


class Person:
    def greet(self) -> str:
        return "hello"


class Rock:
    pass


for thing in [Dog(), Person(), Rock()]:
    print(type(thing).__name__, isinstance(thing, Greetable))

In [ ]:
# In protoform, CharTokenizer satisfies Tokenizer without importing it
from protoform.protocols import Tokenizer
from protoform.tokenizer import CharTokenizer

tok = CharTokenizer("hello world")
print(isinstance(tok, Tokenizer))  # True — shape matches, no inheritance needed

# This means load_encoded accepts any object with encode/decode/vocab_size —
# not just CharTokenizer. Swap in a BPE tokenizer tomorrow and nothing breaks.

## PEP 695 generics — `class Foo[T]`

**What it does:** parameterises a class or protocol over a type using
a clean bracket syntax introduced in Python 3.12.

### Naming conventions

The letter inside the brackets is just a **label for the slot** — it has
no meaning to Python.  By convention:

| Name | When to use it |
|---|---|
| `T` | generic catch-all — *"some type"* |
| `K`, `V` | key / value in dict-like structures |
| `CorpusT`, `ModelT` | domain-specific — name the slot after what goes in it |

We use `CorpusT` in `protoform` because it tells you immediately what the
slot is for — no guessing:

```python
class Tokenizer[CorpusT: Corpus](Protocol):
    def encode(self, data: CorpusT) -> list[int]: ...
    def decode(self, ids: list[int]) -> CorpusT: ...
```

The `: Corpus` **bounds** `CorpusT` — it means *CorpusT must be a subtype
of Corpus*.  Once a caller writes `Tokenizer[str]`, every `CorpusT` in
that class becomes `str`: `encode` must receive a `str`, `decode` must
return a `str`.  Pass the wrong type and pyright flags it immediately.

In [ ]:
# --- Python 3.11 style (still valid, but verbose) ---
from typing import Protocol, TypeVar

CorpusT_old = TypeVar("CorpusT_old")


class OldTokenizer(Protocol[CorpusT_old]):
    def encode(self, data: CorpusT_old) -> list[int]: ...
    def decode(self, ids: list[int]) -> CorpusT_old: ...


# --- Python 3.12+ style (PEP 695) ---
# CorpusT is declared inline — no separate TypeVar needed
# The name CorpusT makes the slot's purpose self-documenting
class NewTokenizer[CorpusT](Protocol):
    def encode(self, data: CorpusT) -> list[int]: ...
    def decode(self, ids: list[int]) -> CorpusT: ...


# Concrete implementation — CorpusT resolves to str
class TextTokenizer:
    def encode(self, data: str) -> list[int]:
        return [ord(c) for c in data]

    def decode(self, ids: list[int]) -> str:
        return "".join(chr(i) for i in ids)


tok = TextTokenizer()
ids = tok.encode("hi")
print("encoded:", ids)
print("decoded:", tok.decode(ids))


# PEP 695 also works on functions — T is fine here, it's truly generic
def first[T](items: list[T]) -> T:
    return items[0]


print(first(["a", "b", "c"]))  # T resolves to str
print(first([1, 2, 3]))  # T resolves to int

### The compile-time payoff

The value of `Tokenizer[CorpusT]` is not in the code above — it runs fine
regardless.  The value is what **pyright catches before you run anything**.

Consider a function that accepts a tokenizer and some data:

```python
def process(tokenizer: Tokenizer[str], text: str) -> list[int]:
    return tokenizer.encode(text)
```

Pyright now knows:
- `tokenizer` must be a `Tokenizer[str]` — `CorpusT` is resolved to `str`
- Passing a `Tokenizer[ImageCorpus]` is a **type error at write time**
- Passing `text=42` is a **type error at write time**

Neither error reaches `python` — your editor underlines it the moment you type it.

```python
# pyright flags this before you run a single line:
process(PassthroughTokenizer(), "hello")
# error: Argument of type 'PassthroughTokenizer' cannot be assigned
#        to parameter 'tokenizer' of type 'Tokenizer[str]'
```

Compare to a dynamically-typed version with no hints:
the same mistake silently reaches production and crashes
with an `AttributeError` or wrong output at 3am.

In [ ]:
# protoform uses PEP 695 syntax with a named, bounded slot:
#   class Tokenizer[CorpusT: Corpus](Protocol)
#
# CorpusT — the name tells you what goes in the slot
# : Corpus — the bound tells you what it must satisfy
from protoform.protocols import Tokenizer
from protoform.tokenizer import CharTokenizer

# CharTokenizer satisfies Tokenizer[str] — CorpusT resolved to str
tok: Tokenizer[str] = CharTokenizer("hello world")

ids = tok.encode("hello")  # encode takes str  (CorpusT = str)
print("encoded:", ids)

text = tok.decode(ids)  # decode returns str (CorpusT = str)
print("decoded:", text)

# A future image tokenizer would be:
#   ImageTokenizer satisfies Tokenizer[ImageCorpus]
#   CorpusT resolves to ImageCorpus
#   encode takes ImageCorpus, decode returns ImageCorpus
#   passing it where Tokenizer[str] is expected: pyright error

### Future stage — `Tokenizer[CorpusT, TokenT]`

The current protocol fixes the token type to `list[int]`.  This is
correct for the discrete transformer we are building — all modalities
(text, images via VQ-VAE, audio via Encodec) map to integer codebook
indices that `nn.Embedding` looks up.

A continuous encoder like CLIP or BERT skips the codebook entirely and
produces float embeddings directly.  That is a different abstraction —
an *encoder*, not a *tokenizer* — and it does not fit `list[int]`.

Once the base transformer is trained on Shakespeare, a later stage of
`protoform` will introduce a second type parameter:

```python
# Future — TokenT makes the token type explicit and flexible
class Tokenizer[CorpusT: Corpus, TokenT](Protocol):
    def encode(self, data: CorpusT) -> list[TokenT]: ...
    def decode(self, ids: list[TokenT]) -> CorpusT: ...
```

This will cover:

| Tokenizer | CorpusT | TokenT |
|---|---|---|
| `CharTokenizer` | `str` | `int` |
| `VQVAETokenizer` | `ImageCorpus` | `int` |
| `EncodecTokenizer` | `AudioCorpus` | `int` |
| `CLIPEncoder` | `ImageCorpus` | `float` |

For now, `list[int]` is not a simplification — it is the right design
for the architecture we are building.